# 직접 해보기 10 · 우주의 자로 팽창률 재기

DESI DR2 BAO 거리비 13개와 공분산(CobayaSampler/bao_data 커밋 b7b8a36, CC BY 4.0)을 읽는다. 같은 계산이 `analysis.py`에도 들어 있다.

In [ ]:
import analysis as A
rows, cov = A.load_bao(), A.load_cov()
print('mean', A.sha8('mean.txt'), 'cov', A.sha8('cov.txt'), len(rows), '줄', len(cov), 'x', len(cov[0]))
for r in rows: print(r)

## 1. 비율을 팽창률로 바꾼다
`H(z) = c ÷ (DH/rd × rd)`, c = 299,792.458 km/s.

In [ ]:
for rd in (147, 140):
    h, e, ratio = A.hubble_at(rows, 0.51, rd)
    print(f'rd={rd}: DH/rd={ratio:.3f} -> H(0.51) = {h:.1f} ± {e:.1f}')

## 2. 자료와 모형을 그려 본다
matplotlib이 없으면 `pip install matplotlib`. 곡선은 평탄 ΛCDM, Ωm=0.30, H₀=68, rd=147.

In [ ]:
import matplotlib.pyplot as plt
zs = [0.02 + 0.02 * i for i in range(125)]
col = {'DM_over_rs': '#2E8C86', 'DH_over_rs': '#D9733D', 'DV_over_rs': '#222'}
plt.figure(figsize=(8, 4.5))
for q, c in col.items():
    plt.plot(zs, [A.model(q, z) for z in zs], color=c, lw=1)
    p = [r for r in rows if r['q'] == q]
    plt.errorbar([r['z'] for r in p], [r['v'] for r in p], [r['s'] for r in p], fmt='o', color=c, label=q)
plt.xlabel('z'); plt.ylabel('distance / rd'); plt.legend(); plt.show()

## 3. χ²를 계산한다
13×13 공분산 전체를 쓴다. 대각 원소만 쓰면 다른 값이 나온다.

In [ ]:
for h0, rd, om in ((68, 147, .30), (71.4, 140, .30), (68, 147, .35)):
    print(f'H0={h0} rd={rd} Om={om}: chi2 = {A.chi2(rows, cov, om, h0, rd):.1f}')

## 4. Ωm을 훑어 본다
이 계산 조건(평탄, 복사·휘어짐·변하는 암흑에너지 없음) 안에서만 의미가 있다.

In [ ]:
oms = [k / 100 for k in range(25, 36)]
c2 = [A.chi2(rows, cov, o, 68, 147) for o in oms]
plt.plot(oms, c2, 'o-'); plt.xlabel('Omega_m'); plt.ylabel('chi2'); plt.show()
print(min(zip(c2, oms)))

## 5. 기록을 남긴다
읽은 값, 정한 조건, 바뀐 결과, 남은 가정을 나누어 적고 파일 지문(mean.txt, cov.txt)을 함께 남긴다.